# Asteroid Family Identification via Hierarchical Clustering Method (HCM)

Implementation of Zappalà et al. 1990 — *Asteroid Families I: Identification by Hierarchical Clustering and Reliability Assessment*

**Method**: Single-linkage hierarchical clustering in proper orbital element space, using a physically motivated velocity-space distance metric (Eq. 2 of the paper).

**Benchmark**: AstDys family catalog (Milani group). Goal: identify ≥8 families at ≥95% completeness.

## 1. Imports and Configuration

In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from mpl_toolkits.mplot3d import Axes3D
from scipy.cluster.hierarchy import linkage, fcluster

%matplotlib widget

# ---------------------------------------------------------------------------
# Zone boundaries from Table I, Zappalà et al. 1990
# ---------------------------------------------------------------------------
ZONE_BOUNDARIES = {
    2: (2.065, 2.3),
    3: (2.3,   2.501),
    4: (2.501, 2.825),
    5: (2.825, 2.958),
    6: (2.958, 3.030),
    7: (3.030, 3.278),
}

# ---------------------------------------------------------------------------
# Per-zone velocity cutoffs (m/s).
# These need to be re-swept after each new dendrogram run.
# Start here; run quick_sweep() to tune for your dataset.
# ---------------------------------------------------------------------------
ZONE_CUTOFFS = {
    2: 60,
    3: 50,
    4: 45,
    5: 50,
    6: 40,
    7: 45,
}

# ---------------------------------------------------------------------------
# Target families — AstDys family IDs and names
#
# Original 10 from Zappalà 1990 + additional well-established families.
# Selection criteria: large (>1000 known members), old/compact, away from
# busy resonance regions, strong signal in proper element space.
#
# Good candidates to add: Flora (4), Nysa (44), Hungaria (434),
#   Hygiea (10), Alauda (702), Euphrosyne (31)
# ---------------------------------------------------------------------------
csv_families = df[df['family1'] > 0]['family1'].value_counts()

FAMILY_NAMES = {
    # --- Validated original families ---
    4: 'Vesta', 
    10: 'Hygiea',
    15: 'Eunomia', 
    20: 'Massalia', 
    24: 'Themis',
    31: 'Euphrosyne', 
    93: 'Minerva', 
    135: 'Hertha',
    158: 'Koronis', 
    163: 'Erigone',
    170: 'Maria', 
    221: 'Eos', 
    302: 'Clarissa', 
    375: 'Ursula', 
    434: 'Hungaria', 
    480: 'Hansa',
    490: 'Veritas', 
    569: 'Misa', 
    668: 'Dora', 
    808: 'Merxia',
    847: 'Agnia', 
    1040: 'Natasha', 
    
    # --- Newly identified large families ---
    3: 'Juno',
    25: 'Phocaea',
    110: 'Lydia',
    145: 'Adeona',
    194: 'Prokne',
    283: 'Emma',
    293: 'Brasilia',
    396: 'Aeolia',
    606: 'Brangäne',
    778: 'Theobalda',
    945: 'Barcelona',
    1547: 'Nele',
    1658: 'Innes',
    1726: 'Hoffmeister',
    1911: 'Schubart',
    2076: 'Levin',
    3330: 'Gantrisch',
    3815: 'König',
    3827: 'Zdeněkhorský',
    10955: 'Harig'
}


In [26]:

print("=" * 55)
print("FAMILIES IN FAMILY_NAMES — present in CSV?")
print("=" * 55)
valid, missing = [], []
for num, name in sorted(FAMILY_NAMES.items()):
    if num in csv_families.index:
        count = csv_families[num]
        valid.append(num)
        print(f"  ✓ {num:>5}  {name:<15}  {count:>7,} members")
    else:
        missing.append(num)
        print(f"  ✗ {num:>5}  {name:<15}  NOT IN CSV — remove from benchmarking")

print(f"\n  Valid: {len(valid)}   Missing/ghost: {len(missing)}")
print(f"  Ghost families to remove: {[FAMILY_NAMES[n] for n in missing]}")

print()
print("=" * 55)
print("LARGE FAMILIES IN CSV — not in FAMILY_NAMES (unknown)")
print("=" * 55)
for num, count in csv_families.items():
    if num not in FAMILY_NAMES and count > 1000:
        print(f"  ?  {num:>5}  {'???':<15}  {count:>7,} members  ← add if identified")


FAMILIES IN FAMILY_NAMES — present in CSV?
  ✓     3  Juno               4,296 members
  ✓     4  Vesta             16,447 members
  ✓    10  Hygiea             4,594 members
  ✓    15  Eunomia           20,216 members
  ✓    20  Massalia          14,324 members
  ✓    24  Themis             9,887 members
  ✓    25  Phocaea            2,744 members
  ✓    31  Euphrosyne         3,876 members
  ✓    93  Minerva            3,873 members
  ✓   110  Lydia              1,450 members
  ✓   135  Hertha            24,796 members
  ✓   145  Adeona             4,088 members
  ✓   158  Koronis           13,240 members
  ✓   163  Erigone            1,644 members
  ✓   170  Maria              6,594 members
  ✓   194  Prokne             1,019 members
  ✓   221  Eos               34,884 members
  ✓   283  Emma               1,012 members
  ✓   293  Brasilia           2,001 members
  ✓   302  Clarissa             486 members
  ✓   375  Ursula             1,864 members
  ✓   396  Aeolia             1,6

## 2. Load Data

In [27]:
# proper_asteroid_data.csv  : proper orbital elements (a, e, sin_i, ...)
# family_membership.csv     : AstDys family assignments (name, family1, ...)
proper   = pd.read_csv('proper_asteroid_data_clean.csv')
families = pd.read_csv('family_membership.csv')

df = proper.merge(families[['name', 'family1']], on='name', how='left')
df['family1'] = df['family1'].fillna(0).astype(int)

print(f"Total asteroids loaded: {len(df):,}")
print(f"Asteroids with family assignment: {(df['family1'] > 0).sum():,}")
print()

# Count members per target family
print("Target family sizes in dataset:")
for fid, fname in FAMILY_NAMES.items():
    n = (df['family1'] == fid).sum()
    print(f"  {fname:12s} (id={fid:4d}): {n:,}")

/tmp/ipykernel_541716/988242616.py:3: DtypeWarning: Columns (0: name) have mixed types. Specify dtype option on import or set low_memory=False.
  proper   = pd.read_csv('proper_asteroid_data_clean.csv')
/tmp/ipykernel_541716/988242616.py:4: DtypeWarning: Columns (0: name, 1: near1) have mixed types. Specify dtype option on import or set low_memory=False.
  families = pd.read_csv('family_membership.csv')


Total asteroids loaded: 1,152,208
Asteroids with family assignment: 248,507

Target family sizes in dataset:
  Vesta        (id=   4): 16,447
  Hygiea       (id=  10): 4,594
  Eunomia      (id=  15): 20,216
  Massalia     (id=  20): 14,324
  Themis       (id=  24): 9,887
  Euphrosyne   (id=  31): 3,876
  Minerva      (id=  93): 3,873
  Hertha       (id= 135): 24,796
  Koronis      (id= 158): 13,240
  Erigone      (id= 163): 1,644
  Maria        (id= 170): 6,594
  Eos          (id= 221): 34,884
  Clarissa     (id= 302): 486
  Ursula       (id= 375): 1,864
  Hungaria     (id= 434): 4,071
  Hansa        (id= 480): 4,504
  Veritas      (id= 490): 6,197
  Misa         (id= 569): 1,221
  Dora         (id= 668): 3,634
  Merxia       (id= 808): 2,373
  Agnia        (id= 847): 7,099
  Natasha      (id=1040): 5,543
  Juno         (id=   3): 4,296
  Phocaea      (id=  25): 2,744
  Lydia        (id= 110): 1,450
  Adeona       (id= 145): 4,088
  Prokne       (id= 194): 1,019
  Emma         (id= 283

## 3. Visualise Known Families

In [28]:
for h in [15, 16, 17]:
    print(f"\nH <= {h}:")
    print(f"  {'Zone':>6} {'N_total':>10} {'N_family':>10} {'Fam%':>6} {'Dist GB':>9} {'Link GB':>9}")
    print(f"  {'-'*55}")
    sub = df[df['H_mag'] <= h]
    for zone, (a_min, a_max) in ZONE_BOUNDARIES.items():
        zdf     = sub[(sub['a'] >= a_min) & (sub['a'] < a_max)]
        n_total = len(zdf)
        n_fam   = (zdf['family1'] > 0).sum()
        n_pairs = n_total * (n_total - 1) // 2
        dist_gb = n_pairs * 4 / 1e9
        link_gb = dist_gb * 2.5
        print(f"  {zone:>6} {n_total:>10,} {n_fam:>10,} {100*n_fam/max(n_total,1):>5.1f}% {dist_gb:>9.1f} {link_gb:>9.1f}")


H <= 15:
    Zone    N_total   N_family   Fam%   Dist GB   Link GB
  -------------------------------------------------------
       2      5,792        548   9.5%       0.1       0.2
       3     11,411      3,405  29.8%       0.3       0.7
       4     29,558      9,055  30.6%       1.7       4.4
       5      5,026      2,333  46.4%       0.1       0.1
       6      7,326      3,599  49.1%       0.1       0.3
       7     32,662     10,292  31.5%       2.1       5.3

H <= 16:
    Zone    N_total   N_family   Fam%   Dist GB   Link GB
  -------------------------------------------------------
       2     14,371      1,281   8.9%       0.4       1.0
       3     30,926      9,008  29.1%       1.9       4.8
       4     72,256     18,421  25.5%      10.4      26.1
       5     10,702      4,360  40.7%       0.2       0.6
       6     14,775      5,648  38.2%       0.4       1.1
       7     69,939     19,662  28.1%       9.8      24.5

H <= 17:
    Zone    N_total   N_family   Fam%   Di

In [29]:
def plot_family(family_id, df=df, family_names=FAMILY_NAMES):
    """
    Plot a known family in all three proper-element planes:
    (a, e), (a, sin_i), (e, sin_i).
    Red = family members, grey = background.
    """
    name    = family_names.get(family_id, f'Family {family_id}')
    members = df[df['family1'] == family_id]
    bg      = df[df['family1'] != family_id]
    print(f"{name}: {len(members):,} members in dataset")

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, (xcol, ycol) in zip(axes, [('a','e'), ('a','sin_i'), ('e','sin_i')]):
        ax.scatter(bg[xcol],      bg[ycol],      s=1, c='lightgray', alpha=0.3, label='background')
        ax.scatter(members[xcol], members[ycol], s=8, c='red',       alpha=0.7,
                   label=f'{name} (n={len(members):,})')
        ax.set_xlabel(xcol); ax.set_ylabel(ycol)
        ax.set_title(f'{name} — {xcol} vs {ycol}')
        ax.set_xlim(members[xcol].min()-0.1, members[xcol].max()+0.1)
        ax.set_ylim(members[ycol].min()-0.1, members[ycol].max()+0.1)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{name.lower()}_ground_truth.png', dpi=150)
    plt.show()


def plot_family_3d(family_id, df=df, family_names=FAMILY_NAMES):
    """3D scatter of a family in (a, e, sin_i) space."""
    name    = family_names.get(family_id, f'Family {family_id}')
    members = df[df['family1'] == family_id]
    bg      = df[df['family1'] != family_id]
    print(f"{name}: {len(members):,} members in dataset")

    fig = plt.figure(figsize=(8, 6))
    ax  = fig.add_subplot(111, projection='3d')
    ax.scatter(bg['a'],      bg['e'],      bg['sin_i'],      s=1, c='lightgray', alpha=0.2, label='background')
    ax.scatter(members['a'], members['e'], members['sin_i'], s=8, c='red',       alpha=0.7,
               label=f'{name} (n={len(members):,})')
    ax.set_xlabel('a (AU)'); ax.set_ylabel('e'); ax.set_zlabel('sin i')
    ax.set_title(f'{name} — 3D proper element space')
    ax.legend()
    plt.tight_layout()
    plt.show()


# Example — uncomment to plot:
# plot_family(158)      # Koronis
# plot_family_3d(24)    # Themis

## 4. HCM Core Functions

### Distance metric (Zappalà et al. 1990, Eq. 2)

$$\delta v = n a' \sqrt{k_1 \left(\frac{\delta a'}{a'}\right)^2 + k_2 (\delta e')^2 + k_3 (\delta \sin i')^2}$$

with $k_1 = 5/4$, $k_2 = 2$, $k_3 = 2$ (standard metric coefficients, Section II of the paper).

$na'$ is the circular orbital velocity at $a'$. Units: AU/yr × 4740.9 = m/s.

In [31]:
# Metric coefficients — Zappalà et al. 1990, Section II
K1, K2, K3 = 5/4, 2, 2

# 1 AU/yr in m/s: (1.496e11 m) / (3.156e7 s) = 4740.9 m/s
AU_YR_TO_MS = 4740.47


def compute_distances(X, chunk_size=1000, verbose=True):
    """
    Compute all N*(N-1)/2 pairwise delta_v distances (Eq. 2) in row chunks
    to avoid allocating an N^2 array all at once.

    For each row i, we vectorise over all j > i simultaneously using numpy,
    so the inner loop is a numpy operation, not a Python loop per pair.
    Peak memory ~ chunk_size * N * 4 bytes (float32).

    Parameters
    ----------
    X          : array (N, 3) — columns [a', e', sin_i']
    chunk_size : rows processed per outer iteration (1000 → ~few hundred MB)

    Returns
    -------
    dist : condensed distance array, length N*(N-1)/2, dtype float32
           Compatible with scipy.cluster.hierarchy.linkage.
    """
    N       = len(X)
    n_pairs = N * (N - 1) // 2
    dist    = np.empty(n_pairs, dtype=np.float32)

    a, e, si = X[:, 0], X[:, 1], X[:, 2]
    pair_idx = 0

    for i_start in range(0, N, chunk_size):
        i_end = min(i_start + chunk_size, N)

        if verbose and i_start % (chunk_size * 20) == 0:
            print(f"    {100*pair_idx/n_pairs:.1f}%  "
                  f"({pair_idx:,} / {n_pairs:,} pairs)", flush=True)

        for i in range(i_start, i_end):
            j = i + 1
            if j >= N:
                continue

            # All pairs (i, j), (i, j+1), ..., (i, N-1) at once
            a0  = (a[i] + a[j:]) / 2.0
            n0  = 2.0 * np.pi / (a0 ** 1.5)   # Kepler: n = 2π / T = 2π / a^1.5
            da  = a[i] - a[j:]
            de  = e[i] - e[j:]
            dsi = si[i] - si[j:]

            dv = n0 * a0 * np.sqrt(
                K1 * (da / a0)**2 +
                K2 * de**2 +
                K3 * dsi**2
            ) * AU_YR_TO_MS

            n_new = len(dv)
            dist[pair_idx : pair_idx + n_new] = dv
            pair_idx += n_new

    return dist


def run_hcm_zone(df, zone, save=True, chunk_size=1000):
    """
    Run HCM on one zone of the asteroid belt.

    Steps:
      1. Filter df to the zone's semimajor axis range.
      2. Compute all pairwise delta_v distances.
      3. Build single-linkage dendrogram (scipy 'single' method).
         Single linkage means: distance(group A, group B) = min pairwise distance.
         This is exactly the update rule in step (3) of the paper.
      4. Cut dendrogram at ZONE_CUTOFFS[zone] to produce family labels.
      5. Return all groups with >= min_members members.

    Saves Z_zone{zone}.npy and zone{zone}_df.csv so you can re-cut
    at different thresholds later without recomputing (use recut_zone).

    Returns
    -------
    families : list of lists of asteroid name strings
    zone_df  : DataFrame for this zone with 'hcm_label' column added
    """
    a_min, a_max  = ZONE_BOUNDARIES[zone]
    cutoff        = ZONE_CUTOFFS[zone]

    zone_df = df[
        (df['a'] >= a_min) & (df['a'] < a_max)
    ].dropna(subset=['a', 'e', 'sin_i']).copy()

    n       = len(zone_df)
    n_pairs = n * (n - 1) // 2
    mem_mb  = n_pairs * 4 / 1e6
    print(f"Zone {zone} ({a_min}–{a_max} AU): {n:,} asteroids | "
          f"{n_pairs:,} pairs | ~{mem_mb:.0f} MB")

    X = zone_df[['a', 'e', 'sin_i']].values

    print(f"  Computing pairwise distances...")
    dist = compute_distances(X, chunk_size=chunk_size)

    print(f"  Building dendrogram (single linkage)...")
    Z = linkage(dist, method='single')

    if save:
        np.save(f'Z_zone{zone}.npy', Z)
        zone_df.to_csv(f'zone{zone}_df.csv', index=False)
        print(f"  Saved Z_zone{zone}.npy + zone{zone}_df.csv")

    labels  = fcluster(Z, t=cutoff, criterion='distance')
    zone_df = zone_df.copy()
    zone_df['hcm_label'] = labels

    families = _labels_to_families(zone_df)
    print(f"  → {len(families)} families at cutoff={cutoff} m/s")
    return families, zone_df


def recut_zone(zone, cutoff, min_members=5):
    """
    Re-cut a previously saved dendrogram at a new cutoff — runs instantly.
    Use this to experiment with cutoffs without recomputing distances.
    """
    Z       = np.load(f'Z_zone{zone}.npy')
    zone_df = pd.read_csv(f'zone{zone}_df.csv')
    labels  = fcluster(Z, t=cutoff, criterion='distance')
    zone_df['hcm_label'] = labels
    families = _labels_to_families(zone_df, min_members)
    largest  = len(families[0]) if families else 0
    print(f"Zone {zone} @ {cutoff} m/s → {len(families)} families, largest={largest:,}")
    return families, zone_df


def _labels_to_families(zone_df, min_members=5):
    """Convert hcm_label column to sorted list of member-name lists."""
    families = []
    for lbl in np.unique(zone_df['hcm_label']):
        members = zone_df[zone_df['hcm_label'] == lbl]['name'].tolist()
        if len(members) >= min_members:
            families.append(members)
    families.sort(key=len, reverse=True)
    return families


def run_all_zones(df, min_members=5):
    """Run HCM across all zones and return all families."""
    all_families = []
    total_start  = time.time()

    for zone in ZONE_BOUNDARIES:
        t0   = time.time()
        fams, _ = run_hcm_zone(df, zone)
        all_families.extend(fams)
        print(f"  Zone {zone} done in {time.time()-t0:.1f}s\n")

    print(f"All zones done in {time.time()-total_start:.1f}s")
    print(f"Total: {len(all_families):,} families, "
          f"{sum(len(f) for f in all_families):,} asteroids assigned")
    return all_families


def recut_all_zones(cutoffs=ZONE_CUTOFFS, min_members=5):
    """
    Re-cut all saved dendrograms using the given cutoff dict.
    Runs in seconds — no distance recomputation needed.
    """
    all_families = []
    for zone, cutoff in cutoffs.items():
        fams, _ = recut_zone(zone, cutoff, min_members)
        all_families.extend(fams)
    print(f"\nTotal: {len(all_families):,} families, "
          f"{sum(len(f) for f in all_families):,} asteroids assigned")
    return all_families

## 5. Run HCM

**First run** (no saved Z files): call `run_all_zones` — computes distances + dendrograms for all zones and saves them to disk.

**Subsequent runs** (Z files already saved): call `recut_all_zones` — loads saved dendrograms and re-cuts instantly.

In [32]:
import os

# Check whether saved Z files exist for all zones
zones_missing = [z for z in ZONE_BOUNDARIES if not os.path.exists(f'Z_zone{z}.npy')]

if zones_missing:
    print(f"No saved dendrograms found for zones {zones_missing}.")
    print("Running full HCM — this may take several minutes...\n")
    all_families = run_all_zones(df)
else:
    print("Saved dendrograms found for all zones. Re-cutting from disk...\n")
    all_families = recut_all_zones()    

Saved dendrograms found for all zones. Re-cutting from disk...

Zone 2 @ 60 m/s → 263 families, largest=773
Zone 3 @ 50 m/s → 245 families, largest=3,840


/tmp/ipykernel_541716/2560165266.py:128: DtypeWarning: Columns (0: name) have mixed types. Specify dtype option on import or set low_memory=False.
  zone_df = pd.read_csv(f'zone{zone}_df.csv')


Zone 4 @ 45 m/s → 673 families, largest=2,163
Zone 5 @ 50 m/s → 60 families, largest=3,250
Zone 6 @ 40 m/s → 109 families, largest=3,828
Zone 7 @ 45 m/s → 791 families, largest=6,587

Total: 2,141 families, 61,577 asteroids assigned


## 6. Evaluate Against AstDys Ground Truth

In [33]:
def evaluate(all_families, df, family_names=FAMILY_NAMES, threshold=0.95):
    """
    Compare HCM output against AstDys ground truth.

    For each known family, find the HCM cluster that contains the most
    true members (voting). Report:
      - Completeness (recall)  = recovered / true_size
      - Precision              = recovered / found_size
      - F1                     = harmonic mean of completeness and precision

    A family 'passes' if completeness >= threshold.
    """
    # Map each asteroid name to the index of its HCM family
    name_to_fi = {}
    for i, fam in enumerate(all_families):
        for name in fam:
            name_to_fi[name] = i

    rows = []
    for fam_id, fam_name in family_names.items():
        true_members = set(df[df['family1'] == fam_id]['name'].tolist())
        if not true_members:
            print(f"  {fam_name}: no members in dataset, skipping")
            continue

        # Vote: which HCM family received the most true members?
        vote = {}
        for name in true_members:
            if name in name_to_fi:
                fi = name_to_fi[name]
                vote[fi] = vote.get(fi, 0) + 1

        if not vote:
            completeness = precision = f1 = 0.0
            recovered = found_size = 0
        else:
            best_fi      = max(vote, key=vote.get)
            recovered    = vote[best_fi]
            found_size   = len(all_families[best_fi])
            completeness = recovered / len(true_members)
            precision    = recovered / found_size
            f1           = (2 * precision * completeness /
                            (precision + completeness)
                            if (precision + completeness) > 0 else 0.0)

        rows.append({
            'family':       fam_name,
            'true_size':    len(true_members),
            'recovered':    recovered,
            'found_size':   found_size,
            'completeness': round(completeness * 100, 1),
            'precision':    round(precision    * 100, 1),
            'f1':           round(f1           * 100, 1),
            'pass':         '✓' if completeness >= threshold else '✗',
        })

    results = pd.DataFrame(rows).sort_values('completeness', ascending=False)

    print(f"  {'Family':12} {'True':>6} {'Recov':>6} {'Found':>7} "
          f"{'Compl':>7} {'Prec':>7} {'F1':>7}  Pass")
    print("-" * 70)
    for _, r in results.iterrows():
        print(f"  {r['family']:12} {r['true_size']:>6,} {r['recovered']:>6,} "
              f"{r['found_size']:>7,} {r['completeness']:>6.1f}% "
              f"{r['precision']:>6.1f}% {r['f1']:>6.1f}%  {r['pass']}")

    n_pass = results['pass'].eq('✓').sum()
    print(f"\n{n_pass}/{len(results)} families reach {int(threshold*100)}% completeness")
    return results


results = evaluate(all_families, df)

  Family         True  Recov   Found   Compl    Prec      F1  Pass
----------------------------------------------------------------------
  Emma          1,012    362     385   35.8%   94.0%   51.8%  ✗
  Minerva       3,873  1,254   1,410   32.4%   88.9%   47.5%  ✗
  Hygiea        4,594  1,483   1,821   32.3%   81.4%   46.2%  ✗
  Lydia         1,450    344     386   23.7%   89.1%   37.5%  ✗
  Natasha       5,543  1,283   1,558   23.1%   82.3%   36.1%  ✗
  Adeona        4,088    923   1,139   22.6%   81.0%   35.3%  ✗
  Dora          3,634    778     848   21.4%   91.7%   34.7%  ✗
  Veritas       6,197  1,320   6,587   21.3%   20.0%   20.7%  ✗
  Innes         1,282    243     274   19.0%   88.7%   31.2%  ✗
  Merxia        2,373    405     445   17.1%   91.0%   28.7%  ✗
  Gantrisch     3,788    649     705   17.1%   92.1%   28.9%  ✗
  Levin         2,365    379     652   16.0%   58.1%   25.1%  ✗
  Zdeněkhorský  1,709    271     284   15.9%   95.4%   27.2%  ✗
  Themis        9,887  1,497  

In [8]:
# Where are Koronis and Massalia members actually located?
for fam_id, fam_name in [(158, 'Koronis'), (20, 'Massalia')]:
    members = df[df['family1'] == fam_id]
    print(f"\n{fam_name} ({len(members):,} members)")
    print(f"  a range: {members['a'].min():.3f} – {members['a'].max():.3f} AU")
    for zone, (a_min, a_max) in ZONE_BOUNDARIES.items():
        n = len(members[(members['a'] >= a_min) & (members['a'] < a_max)])
        if n > 0:
            # Check how many are in the saved zone CSV
            zdf = pd.read_csv(f'zone{zone}_df.csv', low_memory=False)
            n_in_csv = len(zdf[zdf['family1'] == fam_id])
            print(f"  Zone {zone}: {n:,} members in full df | {n_in_csv:,} in zone CSV")


Koronis (13,240 members)
  a range: 2.816 – 2.986 AU
  Zone 4: 2 members in full df | 0 in zone CSV
  Zone 5: 13,172 members in full df | 3,441 in zone CSV
  Zone 6: 66 members in full df | 21 in zone CSV

Massalia (14,324 members)
  a range: 2.334 – 2.474 AU
  Zone 3: 14,324 members in full df | 639 in zone CSV


In [9]:
print(ZONE_CUTOFFS)

{2: 60, 3: 50, 4: 45, 5: 50, 6: 40, 7: 45}


In [10]:
for zone in [2,3,4,5,6,7]:
    zdf = pd.read_csv(f'zone{zone}_df.csv')
    print(f"Zone {zone}: {len(zdf):,} asteroids | H range: {zdf['H_mag'].min():.1f} – {zdf['H_mag'].max():.1f}")

Zone 2: 14,371 asteroids | H range: 6.5 – 16.0
Zone 3: 30,926 asteroids | H range: 3.3 – 16.0
Zone 4: 72,256 asteroids | H range: 3.4 – 16.0
Zone 5: 10,702 asteroids | H range: 6.0 – 16.0
Zone 6: 14,775 asteroids | H range: 7.1 – 16.0
Zone 7: 69,939 asteroids | H range: 5.4 – 16.0


/tmp/ipykernel_180892/32433804.py:2: DtypeWarning: Columns (0: name) have mixed types. Specify dtype option on import or set low_memory=False.
  zdf = pd.read_csv(f'zone{zone}_df.csv')


In [11]:
koronis_zone = 5  # Koronis is in zone 5 (2.825–2.958 AU)
zdf = pd.read_csv(f'zone5_df.csv')
koronis_members = zdf[zdf['family1'] == 158]
print(f"Koronis members in zone5_df: {len(koronis_members)}")
print(koronis_members[['a','e','sin_i']].describe())

Koronis members in zone5_df: 3441
                 a            e        sin_i
count  3441.000000  3441.000000  3441.000000
mean      2.893257     0.054480     0.036729
std       0.034566     0.015932     0.002125
min       2.827320     0.016520     0.029060
25%       2.864936     0.043932     0.035483
50%       2.894277     0.049894     0.036637
75%       2.923643     0.065185     0.037634
max       2.956467     0.100948     0.047237


In [12]:
# for zone in [2, 3, 4, 5, 6, 7]:
#     print()
#     sweep_with_truth(zone, [5, 10, 15, 20, 25, 30, 40, 50, 60, 80], df, FAMILY_NAMES)

In [13]:
# Safely convert strings to numbers; non-numeric strings become NaN
numeric_names = pd.to_numeric(df['name'], errors='coerce')

# Filter for numbered asteroids <= 4265 (NaNs are automatically dropped)
sub = df[numeric_names <= 4265].copy()

print(f"Total: {len(sub):,}")

for zone, (a_min, a_max) in ZONE_BOUNDARIES.items():
    zdf = sub[(sub['a'] >= a_min) & (sub['a'] < a_max)]
    n_fam = (zdf['family1'] > 0).sum()
    print(f"Zone {zone}: {len(zdf):,} total | {n_fam:,} family ({100*n_fam/max(len(zdf),1):.1f}%)")

Total: 4,040
Zone 2: 678 total | 25 family (3.7%)
Zone 3: 676 total | 136 family (20.1%)
Zone 4: 1,031 total | 274 family (26.6%)
Zone 5: 290 total | 156 family (53.8%)
Zone 6: 297 total | 214 family (72.1%)
Zone 7: 891 total | 328 family (36.8%)


In [14]:
def quick_sweep(zone, cutoffs):
    Z      = np.load(f'Z_zone{zone}.npy')
    zdf    = pd.read_csv(f'zone{zone}_df.csv', low_memory=False)
    N      = len(zdf)
    labels = fcluster(Z, t=max(cutoffs), criterion='distance')
    
    print(f"\nZone {zone} — {N:,} asteroids")
    print(f"  {'Cutoff':>8} {'N_clusters':>12} {'Largest':>10} {'2nd':>10}")
    print(f"  {'-'*44}")
    for c in cutoffs:
        labs       = fcluster(Z, t=c, criterion='distance')
        sizes      = np.bincount(labs)
        sizes_sort = np.sort(sizes)[::-1]
        n_clusters = (sizes > 1).sum()
        print(f"  {c:>8} {n_clusters:>12,} {sizes_sort[0]:>10,} {sizes_sort[1]:>10,}")

for zone in [2,3,4,5,6,7]:
    quick_sweep(zone, [5,10,15,20,25,30,40,50,60,80,100,120])


Zone 2 — 14,371 asteroids
    Cutoff   N_clusters    Largest        2nd
  --------------------------------------------
         5           11          2          2
        10           47          3          3
        15          158          6          5
        20          346         12          7
        25          565         17          8
        30          828         42         10
        40        1,387        182         20
        50        1,759        373        341
        60        1,821        773        652
        80        1,309      4,082      1,333
       100          859      8,807        145
       120          642     10,392         69

Zone 3 — 30,926 asteroids
    Cutoff   N_clusters    Largest        2nd
  --------------------------------------------
         5           48          2          2
        10          271          4          3
        15          737          6          5
        20        1,429         10          8
        25        2,058 

In [15]:
def sweep_with_truth(zone, cutoffs, df, family_names, min_members=5):
    """
    Sweep cutoffs for a zone and show recovery of known families at each level.
    Much more informative than just looking at cluster sizes.
    """
    Z       = np.load(f'Z_zone{zone}.npy')
    zone_df = pd.read_csv(f'zone{zone}_df.csv')

    # Which known families live in this zone?
    a_min, a_max = ZONE_BOUNDARIES[zone]
    zone_family_ids = []
    for fam_id, fam_name in family_names.items():
        members_in_zone = df[
            (df['family1'] == fam_id) &
            (df['a'] >= a_min) & (df['a'] < a_max)
        ]
        if len(members_in_zone) > 0:
            zone_family_ids.append((fam_id, fam_name, len(members_in_zone)))

    if not zone_family_ids:
        print(f"Zone {zone}: no known families")
        return

    print(f"\nZone {zone} ({a_min}–{a_max} AU)")
    print(f"Known families in this zone: "
          f"{', '.join(f'{n}({c})' for _,n,c in zone_family_ids)}")
    print()

    # Header
    header = f"{'Cutoff':>8} {'Largest':>8}"
    for _, fname, ftrue in zone_family_ids:
        header += f"  {fname[:8]:>10}({ftrue})"
    print(header)
    print("-" * len(header))

    for c in cutoffs:
        labels = fcluster(Z, t=c, criterion='distance')
        zone_df['hcm_label'] = labels

        # For each known family, find best matching cluster
        row = f"{c:>8} "

        # largest cluster size
        _, counts = np.unique(labels, return_counts=True)
        row += f"{counts.max():>8}"

        for fam_id, fname, ftrue in zone_family_ids:
            # merge ground truth into zone_df
            true_names = set(df[df['family1'] == fam_id]['name'].tolist())

            # vote for best cluster
            vote = {}
            for name in true_names:
                match = zone_df[zone_df['name'] == name]
                if len(match) > 0:
                    lbl = match['hcm_label'].iloc[0]
                    vote[lbl] = vote.get(lbl, 0) + 1

            if vote:
                best_lbl     = max(vote, key=vote.get)
                recovered    = vote[best_lbl]
                cluster_size = (zone_df['hcm_label'] == best_lbl).sum()
                pct          = recovered / ftrue * 100
                row += f"  {recovered:>4}/{ftrue:<4} {pct:>4.0f}% (sz={cluster_size})"
            else:
                row += f"  {'—':>14}"

        print(row)

## 7. Visualise HCM Results vs Ground Truth

In [16]:
import os
for zone in [2,3,4,5,6,7]:
    path = f'zone{zone}_df.csv'
    mtime = os.path.getmtime(path)
    import datetime
    print(f"zone{zone}_df.csv — modified: {datetime.datetime.fromtimestamp(mtime)}, "
          f"size: {os.path.getsize(path)/1e6:.1f} MB")

zone2_df.csv — modified: 2026-03-10 15:57:59.959395, size: 1.2 MB
zone3_df.csv — modified: 2026-03-10 15:57:59.970395, size: 2.5 MB
zone4_df.csv — modified: 2026-03-10 15:57:59.978395, size: 6.0 MB
zone5_df.csv — modified: 2026-03-10 15:57:59.988395, size: 0.9 MB
zone6_df.csv — modified: 2026-03-10 15:57:59.992395, size: 1.2 MB
zone7_df.csv — modified: 2026-03-10 15:58:00.010396, size: 5.9 MB


In [17]:
ZONE_CUTOFFS = {2: 60, 3: 35, 4: 55, 5: 20, 6: 27, 7: 45}
all_families = recut_all_zones()
results = evaluate(all_families, df)

Zone 2 @ 60 m/s → 263 families, largest=773
Zone 3 @ 50 m/s → 245 families, largest=3,840


/tmp/ipykernel_180892/2759668535.py:128: DtypeWarning: Columns (0: name) have mixed types. Specify dtype option on import or set low_memory=False.
  zone_df = pd.read_csv(f'zone{zone}_df.csv')


Zone 4 @ 45 m/s → 673 families, largest=2,163
Zone 5 @ 50 m/s → 60 families, largest=3,250
Zone 6 @ 40 m/s → 109 families, largest=3,828
Zone 7 @ 45 m/s → 791 families, largest=6,587

Total: 2,141 families, 61,577 asteroids assigned
  Astraea: no members in dataset, skipping
  Nysa: no members in dataset, skipping
  Alauda: no members in dataset, skipping
  Family         True  Recov   Found   Compl    Prec      F1  Pass
----------------------------------------------------------------------
  Minerva       3,873  1,254   1,410   32.4%   88.9%   47.5%  ✗
  Hygiea        4,594  1,483   1,821   32.3%   81.4%   46.2%  ✗
  Natasha       5,543  1,283   1,558   23.1%   82.3%   36.1%  ✗
  Dora          3,634    778     848   21.4%   91.7%   34.7%  ✗
  Veritas       6,197  1,320   6,587   21.3%   20.0%   20.7%  ✗
  Merxia        2,373    405     445   17.1%   91.0%   28.7%  ✗
  Themis        9,887  1,497   1,507   15.1%   99.3%   26.3%  ✗
  Agnia         7,099    733     770   10.3%   95.2%   1

In [18]:
sweep_with_truth(2, [30, 40, 50, 60, 70, 80], df, FAMILY_NAMES)
sweep_with_truth(5, [5, 10, 15, 20, 25, 30], df, FAMILY_NAMES)


Zone 2 (2.065–2.3 AU)
Known families in this zone: Vesta(2330)

  Cutoff  Largest       Vesta(2330)
-----------------------------------
      30       42     8/2330    0% (sz=8)
      40      182    19/2330    1% (sz=20)
      50      373   296/2330   13% (sz=373)
      60      773   460/2330   20% (sz=773)
      70     2334   475/2330   20% (sz=1039)
      80     4082   475/2330   20% (sz=1333)

Zone 5 (2.825–2.958 AU)
Known families in this zone: Koronis(13172), Eos(90)

  Cutoff  Largest     Koronis(13172)         Eos(90)
----------------------------------------------------
       5      120               —               —
      10      203               —               —
      15      273               —               —
      20      489               —               —
      25      936               —               —
      30     1299               —               —


In [19]:
# def plot_hcm_vs_truth(family_id, all_families, df,
#                       family_names=FAMILY_NAMES):
#     """
#     Side-by-side comparison: AstDys ground truth (top row) vs
#     best-matching HCM cluster (bottom row) in all three element planes.
#     """
#     name         = family_names.get(family_id, f'Family {family_id}')
#     true_members = df[df['family1'] == family_id]
#     true_set     = set(true_members['name'].tolist())

#     # Find the best-matching HCM family by vote
#     name_to_fi = {}
#     for i, fam in enumerate(all_families):
#         for n_ in fam:
#             name_to_fi[n_] = i
#     vote = {}
#     for n_ in true_set:
#         if n_ in name_to_fi:
#             fi = name_to_fi[n_]
#             vote[fi] = vote.get(fi, 0) + 1
#     if not vote:
#         print(f"{name}: no HCM match found")
#         return
#     best_fi     = max(vote, key=vote.get)
#     hcm_names   = set(all_families[best_fi])
#     hcm_members = df[df['name'].isin(hcm_names)]

#     recovered   = len(true_set & hcm_names)
#     completeness = recovered / len(true_set) * 100
#     precision    = recovered / len(hcm_names) * 100
#     print(f"{name}: AstDys={len(true_set):,}  HCM={len(hcm_names):,}  "
#           f"completeness={completeness:.1f}%  precision={precision:.1f}%")

#     plots = [('a','e'), ('a','sin_i'), ('e','sin_i')]
#     fig, axes = plt.subplots(2, 3, figsize=(16, 10))

#     for col, (xcol, ycol) in enumerate(plots):
#         xlim = (true_members[xcol].min()-0.05, true_members[xcol].max()+0.05)
#         ylim = (true_members[ycol].min()-0.05, true_members[ycol].max()+0.05)

#         for row, (data, color, label, title_prefix) in enumerate([
#             (true_members, 'red',  f'AstDys (n={len(true_members):,})', 'Ground Truth'),
#             (hcm_members,  'blue', f'HCM (n={len(hcm_members):,})',     'HCM Result'),
#         ]):
#             ax = axes[row, col]
#             ax.scatter(df[xcol], df[ycol], s=1, c='lightgray', alpha=0.2)
#             ax.scatter(data[xcol], data[ycol], s=5, c=color, alpha=0.7, label=label)
#             ax.set_xlim(*xlim); ax.set_ylim(*ylim)
#             ax.set_xlabel(xcol); ax.set_ylabel(ycol)
#             ax.set_title(f'{name} {title_prefix} — {xcol} vs {ycol}')
#             ax.legend(fontsize=8)

#     plt.suptitle(f'{name}: completeness={completeness:.1f}%  precision={precision:.1f}%',
#                  fontsize=13, y=1.01)
#     plt.tight_layout()
#     plt.savefig(f'{name.lower()}_hcm_vs_truth.png', dpi=150, bbox_inches='tight')
#     plt.show()


# # Plot all target families
# for fid in FAMILY_NAMES:
#     plot_hcm_vs_truth(fid, all_families, df)